<a href="https://colab.research.google.com/github/Tharul-Fernando/CM2604-Telco-Customer-Churn/blob/main/02_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

# Load Original Dataset
file_path = ("/content/drive/MyDrive/ML_Coursework/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(file_path)

# Keep the original data unchanged
df_clean = df.copy()

print("Dataset shape:", df_clean.shape)
display(df_clean.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Convert TotalCharges from text to numerical values
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# Check the new data type
print("Data types:", df_clean["TotalCharges"].dtypes)

# Count missing values after conversion
print("Missing TotalCharges:", df_clean["TotalCharges"].isnull().sum())

Data types: float64
Missing TotalCharges: 11


In [4]:
# Examine customers with missing values
missing_rows = df_clean[df_clean["TotalCharges"].isnull()]

print("Number of affected customers:", len(missing_rows))

print("Tenure values:", missing_rows["tenure"].unique())

display(missing_rows[["tenure","MonthlyCharges","TotalCharges","Churn"]])

Number of affected customers: 11
Tenure values: [0]


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [5]:
# Replace missing TotalCharges values with zero
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)

# Verify that no missing values remain
print("Missing TotalCharges:", df_clean["TotalCharges"].isnull().sum())

# Verify the datatype
print("Data type:",df_clean["TotalCharges"].dtypes)

# Check the dataset dimensions
print("Dataset shape:", df_clean.shape)

Missing TotalCharges: 0
Data type: float64
Dataset shape: (7043, 21)


### Preprocessing 1: Handling Missing TotalCharges Values

**Problem:** The TotalCharges column was stored as an object and contained 11 blank values, even though the initial null-value check detected no missing data.

**Action:** The column was converted to a numerical data type using `pd.to_numeric()` with `errors="coerce"`. The resulting 11 missing values `(NaN)` were then replaced with zero using `fillna(0)`.

**Justification:** All 11 affected customers had a tenure of zero months, suggesting they may not have accumulated any charges. Replacing the missing values with zero was considered a reasonable assumption, allowing all 7,043 customer records to be retained.

In [6]:
# Separate the input features and target variable
X = df_clean.drop(columns = ["customerID","Churn"])
y = df_clean["Churn"]

# Check the dimensions
print("Features (X):", X.shape)
print("Target (y):", y.shape)

# Check the feature names
print("\nFeature names:")
print(X.columns.tolist())

# Check the target classes
print("\nTarget classes:")
print(y.unique())

Features (X): (7043, 19)
Target (y): (7043,)

Feature names:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Target classes:
['No' 'Yes']


In [7]:

# Convert Churn to binary values
y = y.map({"No": 0, "Yes": 1})

# Verify the conversion
print(y.value_counts())

# Check for missing values
print("\nMissing target values:", y.isnull().sum())


Churn
0    5174
1    1869
Name: count, dtype: int64

Missing target values: 0


### Preprocessing 2: Feature and Target Preparation

**Problem:** The dataset contained a unique customer identifier and a categorical target variable that required preparation before model training.

**Action:** The `customerID` column was excluded from the input features. The `Churn` column was separated as the target variable and encoded using 0 for `No` and 1 for `Yes`.

**Justification:** Customer IDs do not provide meaningful information for predicting churn. Excluding them prevents models from learning arbitrary patterns, while binary encoding provides a consistent numerical representation of the target classes that can be used to calculate evaluation metrics later.

In [8]:
# Import the function used to split the dataset
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )

# Display the dimension of each set
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [9]:
# Check the churn distribution in the training set
print("Training set churn distribution:")
print(y_train.value_counts(normalize=True) * 100)

print()

# Check the churn distribution in the testing set
print("Testing set churn distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training set churn distribution:
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing set churn distribution:
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


### Preprocessing 3: Train/Test Splitting

**Problem:** The dataset needed to be divided into separate training and testing sets to evaluate how well the machine learning models perform on unseen customer records.

**Action:** The dataset was split using `train_test_split()` from Scikit-learn, allocating 80% of the records for training and 20% for testing. A fixed `random_state=42` was used for reproducibility, and `stratify=y` was applied to preserve the churn-class proportions.

**Justification:** Separating training and testing data allows the models to be evaluated on records that were not used during training. Stratification helps maintain a similar distribution of churned and non-churned customers in both sets, supporting a more representative evaluation.

In [10]:
# Get categorical columns
cat_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# SeniorCitizen is categorical
cat_cols.append("SeniorCitizen")

# Get numerical columns
num_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Remove SeniorCitizen from numerical columns
num_cols.remove("SeniorCitizen")

print("Categorical columns:", cat_cols)
print("\nNumerical columns:", num_cols)


Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']

Numerical columns: ['tenure', 'MonthlyCharges', 'TotalCharges']


In [11]:
# Import Encoding Libraries
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [12]:
# Create the encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apply encoding to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", encoder, cat_cols),
        ("num", "passthrough", num_cols)
    ],
    verbose_feature_names_out=False
)


In [13]:
# Encode training data
train_enc = preprocessor.fit_transform(X_train)

# Encode testing data
test_enc = preprocessor.transform(X_test)

print("Training:", train_enc.shape)
print("Testing:", test_enc.shape)


Training: (5634, 46)
Testing: (1409, 46)


In [14]:
# Get encoded column names
cols = preprocessor.get_feature_names_out()

# Create a readable table
preview = pd.DataFrame(
    train_enc,
    columns=cols,
    index=X_train.index
)

# Display first five rows
display(preview.head())

# Check missing values
print("Missing values:", pd.isnull(train_enc).sum())

# Count features
print("Total features:", len(cols))


,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,...,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen_0,SeniorCitizen_1,tenure,MonthlyCharges,TotalCharges
3738,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,35.0,49.20,1701.65
3151,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,15.0,75.10,1151.55
4860,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,13.0,40.55,590.35
3867,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,26.0,73.50,1905.70
3810,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,44.55,44.55


Missing values: 0
Total features: 46


### Preprocessing 4: Categorical Feature Encoding

**Problem:** The dataset contained categorical features such as `Contract`, `InternetService` and `PaymentMethod` that needed to be converted into numerical values before model training.

**Action:** Categorical and numerical features were identified separately. `SeniorCitizen` was treated as categorical because it represents a binary category. One-hot encoding was applied using `OneHotEncoder` and `ColumnTransformer`, while the numerical features were left unchanged.

**Justification:** One-hot encoding converts categories into numerical representations without assigning an artificial ordering. The encoder was fitted using only the training data, and the same transformation was applied to the testing data to prevent test-data leakage.